<a href="https://colab.research.google.com/github/misbahhassan6400/flyrank-ml-internship/blob/main/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
!pip -q install duckdb huggingface_hub pandas scikit-learn scipy

import duckdb
import pandas as pd
import numpy as np
import json
import os

from google.colab import userdata

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.compose import TransformedTargetRegressor
from sklearn.metrics import mean_absolute_error
from scipy.stats import spearmanr

HF_TOKEN = userdata.get("HF_TOKEN")
if not HF_TOKEN:
    raise ValueError("HF_TOKEN missing. Colab Secrets mein access ON karo.")

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
)
""")

files = con.sql("""
SELECT file
FROM glob('hf://datasets/FlyRank/internship-warehouse/**/*.parquet')
""").df()

feb_matches = files[
    files["file"].astype(str).str.contains("2026-02", regex=False)
]

mar_matches = files[
    files["file"].astype(str).str.contains("2026-03", regex=False)
]

FEB = feb_matches.iloc[0]["file"]
MAR = mar_matches.iloc[0]["file"]

print("Setup complete")
print("FEB:", FEB)
print("MAR:", MAR)

Setup complete
FEB: hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/data_0.parquet
MAR: hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet


## 1. Method choice and why

I chose a Random Forest Regressor with a log-transformed March GA4 sessions target. The target is numeric and heavy-tailed, so the log transformation reduces the influence of a small number of very large pages. Random Forest can capture nonlinear relationships between impressions, clicks, position, and sessions without assuming a straight-line relationship.

The model is used for directional decision-support, not causal explanation. The model will be compared with the Week-4 rule-based baseline on the same rows, same held-out client groups, and same ranking metrics.

### Research paper note

I read the assigned paper. Its main modeling or evaluation idea was: [write this in your own words]. One limitation or assumption I noticed was: [write this in your own words].

In [ ]:
model_df = con.sql(f"""
WITH feb AS (
    SELECT
        client_hash_id,
        content_hash_id,

        COALESCE(
            SUM(gsc_impressions)
            FILTER (WHERE gsc_data_available IS TRUE),
            0
        ) AS gsc_impressions,

        COALESCE(
            SUM(gsc_clicks)
            FILTER (WHERE gsc_data_available IS TRUE),
            0
        ) AS gsc_clicks,

        AVG(gsc_avg_position)
            FILTER (
                WHERE gsc_data_available IS TRUE
                AND gsc_avg_position IS NOT NULL
            ) AS gsc_avg_position,

        COALESCE(
            SUM(ga4_sessions)
            FILTER (WHERE ga4_data_available IS TRUE),
            0
        ) AS ga4_sessions,

        COALESCE(
            SUM(ga4_engaged_sessions)
            FILTER (WHERE ga4_data_available IS TRUE),
            0
        ) AS ga4_engaged_sessions

    FROM read_parquet('{FEB}')
    WHERE month = '2026-02'
    GROUP BY client_hash_id, content_hash_id
),

mar AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(ga4_sessions)
            FILTER (WHERE ga4_data_available IS TRUE)
            AS march_ga4_sessions

    FROM read_parquet('{MAR}')
    WHERE month = '2026-03'
    GROUP BY client_hash_id, content_hash_id
)

SELECT
    feb.client_hash_id,
    feb.content_hash_id,
    feb.gsc_impressions,
    feb.gsc_clicks,
    feb.gsc_avg_position,
    feb.ga4_sessions,
    feb.ga4_engaged_sessions,
    mar.march_ga4_sessions

FROM feb
INNER JOIN mar
    USING (client_hash_id, content_hash_id)

WHERE mar.march_ga4_sessions IS NOT NULL
""").df()

feature_cols = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "ga4_engaged_sessions"
]

X = model_df[feature_cols].copy()
y = model_df["march_ga4_sessions"].astype(float)
groups = model_df["client_hash_id"].fillna("MISSING_CLIENT")

print("Rows:", len(model_df))
print("Features:", feature_cols)
display(model_df.head())

assert len(feature_cols) == 5
assert "march_ga4_sessions" not in feature_cols


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 80877
Features: ['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_sessions', 'ga4_engaged_sessions']


,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,ga4_engaged_sessions,march_ga4_sessions
0,client_e547b89c05043229,content_9abd8b303f805847,733.0,6.0,6.495085,6.0,0.0,3.0
1,client_e547b89c05043229,content_5f58c55cbfee172a,514.0,0.0,10.490023,1.0,0.0,1.0
2,client_e547b89c05043229,content_6fe390ba3af1e456,2931.0,3.0,38.436254,6.0,1.0,4.0
3,client_e547b89c05043229,content_3ad5d2160242b9ca,970.0,2.0,9.710810,3.0,0.0,1.0
4,client_e547b89c05043229,content_a2bd730a7cf68316,551.0,1.0,6.017373,3.0,0.0,1.0


## 2. Split design

I use a grouped train-test split by client. This keeps all content from a client in either train or test, instead of allowing the same client to appear in both. This is more honest for a new-client or cross-client decision setting and prevents client-specific patterns from leaking across the split.

The Week-4 baseline does not train, so I apply its rule-based score to the same held-out client groups used for the model.

In [ ]:
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

print("Train rows:", len(train_idx))
print("Test rows:", len(test_idx))
print(
    "Train clients:",
    groups.iloc[train_idx].nunique()
)
print(
    "Test clients:",
    groups.iloc[test_idx].nunique()
)

assert len(set(groups.iloc[train_idx]).intersection(
    set(groups.iloc[test_idx])
)) == 0

print("Grouped split check: PASS")

Train rows: 66273
Test rows: 14604
Train clients: 28
Test clients: 8
Grouped split check: PASS


## 3. Train + compare vs my baseline

The W04 baseline prioritizes top-ten pages with low CTR as CTR-fix candidates, then high-impression pages ranking from positions 11 to 20 as quick wins. All remaining rows receive a monitor score.

I use Spearman correlation as the main metric because both the baseline and model produce rankings. I also report top-decile recall, which measures how many of the highest-session rows were captured in the highest-ranked decile.

In [ ]:
eval_df = model_df.copy()

eval_df["ctr"] = np.where(
    eval_df["gsc_impressions"] > 0,
    eval_df["gsc_clicks"] / eval_df["gsc_impressions"],
    np.nan
)

eval_df["baseline_reason"] = np.select(
    [
        (
            (eval_df["gsc_avg_position"] <= 10)
            & (eval_df["ctr"] < 0.02)
            & (eval_df["gsc_impressions"] > 0)
        ),
        (
            (eval_df["gsc_impressions"] >= 1000)
            & (eval_df["gsc_avg_position"] > 10)
            & (eval_df["gsc_avg_position"] <= 20)
        )
    ],
    [
        "CTR_FIX",
        "QUICK_WIN_VOLUME"
    ],
    default="MONITOR"
)

eval_df["baseline_score"] = np.select(
    [
        eval_df["baseline_reason"] == "CTR_FIX",
        eval_df["baseline_reason"] == "QUICK_WIN_VOLUME"
    ],
    [
        100 + np.minimum(
            30,
            np.log1p(eval_df["gsc_impressions"])
        ),
        60 + np.minimum(
            30,
            np.log1p(eval_df["gsc_impressions"])
        )
    ],
    default=np.minimum(
        20,
        np.log1p(eval_df["gsc_impressions"])
    )
)

baseline_test_score = eval_df.iloc[test_idx]["baseline_score"].to_numpy()
y_test = y.iloc[test_idx].to_numpy()

print("Baseline test rows:", len(baseline_test_score))
display(
    eval_df.iloc[test_idx][
        ["baseline_score", "baseline_reason", "march_ga4_sessions"]
    ].head()
)


Baseline test rows: 14604


,baseline_score,baseline_reason,march_ga4_sessions
0,106.598509,CTR_FIX,3.0
1,6.244167,MONITOR,1.0
2,7.983440,MONITOR,4.0
3,106.878326,CTR_FIX,1.0
4,106.313548,CTR_FIX,1.0


In [ ]:
rf_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "model",
        RandomForestRegressor(
            n_estimators=150,
            min_samples_leaf=10,
            random_state=42,
            n_jobs=-1
        )
    )
])

model = TransformedTargetRegressor(
    regressor=rf_pipeline,
    func=np.log1p,
    inverse_func=np.expm1
)

model.fit(
    X.iloc[train_idx],
    y.iloc[train_idx]
)

model_test_pred = model.predict(X.iloc[test_idx])
model_test_pred = np.maximum(model_test_pred, 0)

print("Model trained")
print("Predictions:", len(model_test_pred))

Model trained
Predictions: 14604


In [ ]:
def safe_spearman(actual, predicted):
    result = spearmanr(actual, predicted)
    return float(result.statistic)

def top_decile_recall(actual, predicted):
    k = max(1, int(np.ceil(len(actual) * 0.10)))

    actual_top = set(
        np.argsort(actual)[-k:]
    )

    predicted_top = set(
        np.argsort(predicted)[-k:]
    )

    return len(actual_top.intersection(predicted_top)) / k

results = pd.DataFrame([
    {
        "method": "W04 baseline",
        "spearman_rho": safe_spearman(
            y_test,
            baseline_test_score
        ),
        "top_decile_recall": top_decile_recall(
            y_test,
            baseline_test_score
        )
    },
    {
        "method": "Random Forest",
        "spearman_rho": safe_spearman(
            y_test,
            model_test_pred
        ),
        "top_decile_recall": top_decile_recall(
            y_test,
            model_test_pred
        )
    }
])

display(results)

os.makedirs("work/outputs", exist_ok=True)

with open(
    "work/outputs/w05_model_metrics.json",
    "w"
) as f:
    json.dump(
        results.to_dict(orient="records"),
        f,
        indent=2
    )

print("Metrics JSON written.")


,method,spearman_rho,top_decile_recall
0,W04 baseline,0.520831,0.618070
1,Random Forest,0.587719,0.638604


Metrics JSON written.


## 4. Errors and interpretation

I inspect the largest absolute errors instead of relying only on the summary metrics. Because March sessions are heavy-tailed, a few high-volume content rows may dominate the error. I also inspect feature importance, but I treat it as association rather than causation.


In [ ]:
error_review = eval_df.iloc[test_idx][
    [
        "client_hash_id",
        "content_hash_id",
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position",
        "ga4_sessions",
        "ga4_engaged_sessions",
        "march_ga4_sessions"
    ]
].copy()

error_review["model_prediction"] = model_test_pred
error_review["absolute_error"] = abs(
    error_review["march_ga4_sessions"]
    - error_review["model_prediction"]
)

error_review = error_review.sort_values(
    "absolute_error",
    ascending=False
)

print("Largest model errors:")
display(error_review.head(10))


Largest model errors:


,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,ga4_engaged_sessions,march_ga4_sessions,model_prediction,absolute_error
60360,client_e547b89c05043229,content_eadb33b5df496f4a,94852.0,1123.0,2.559384,577.0,36.0,2730.0,574.316700,2155.683300
62977,client_e547b89c05043229,content_3508adb0f05ec0b9,44487.0,291.0,3.452090,293.0,22.0,191.0,526.323912,335.323912
62899,client_e547b89c05043229,content_3e6275ef355c9602,40080.0,260.0,2.514103,303.0,17.0,194.0,527.984494,333.984494
713,client_e547b89c05043229,content_a6b170f18f827a7f,24105.0,431.0,4.143416,412.0,31.0,521.0,233.464183,287.535817
19836,client_e547b89c05043229,content_c9a0c2fdbdbfb562,142215.0,1605.0,1.882000,395.0,14.0,293.0,570.380987,277.380987
19965,client_e547b89c05043229,content_0e03de7680314cd5,28237.0,162.0,3.141285,102.0,4.0,465.0,187.857621,277.142379
62592,client_e547b89c05043229,content_963de14b1f58978f,30485.0,49.0,4.849475,36.0,0.0,354.0,82.025787,271.974213
65517,client_e547b89c05043229,content_60ffc60f92ae4b26,63169.0,165.0,3.149420,147.0,8.0,64.0,312.742028,248.742028
60906,client_e547b89c05043229,content_ec2e0346994fb5a5,119854.0,490.0,2.594817,450.0,30.0,816.0,574.316700,241.683300
40826,client_e547b89c05043229,content_4078e96bf165d4ef,40183.0,74.0,7.011556,119.0,1.0,60.0,276.414102,216.414102


In [ ]:
rf_importance = pd.Series(
    model.regressor_.named_steps["model"].feature_importances_,
    index=feature_cols
).sort_values(
    ascending=False
)

print("Random Forest feature importance:")
display(rf_importance.to_frame("importance"))

Random Forest feature importance:


,importance
ga4_sessions,0.499761
gsc_impressions,0.302260
gsc_clicks,0.106009
gsc_avg_position,0.089941
ga4_engaged_sessions,0.002028


In [ ]:
error_review["actual_bucket"] = pd.qcut(
    error_review["march_ga4_sessions"].rank(method="first"),
    q=4,
    labels=[
        "lowest_quartile",
        "lower_middle",
        "upper_middle",
        "highest_quartile"
    ]
)

error_summary = (
    error_review
    .groupby("actual_bucket", observed=False)
    .agg(
        n=("absolute_error", "size"),
        median_actual=("march_ga4_sessions", "median"),
        median_absolute_error=("absolute_error", "median")
    )
    .reset_index()
)

display(error_summary)

,actual_bucket,n,median_actual,median_absolute_error
0,lowest_quartile,3651,1.0,1.655672
1,lower_middle,3651,1.0,1.059536
2,upper_middle,3651,2.0,1.435415
3,highest_quartile,3651,9.0,10.867639


The largest errors are concentrated in [write the bucket or pattern you observed]. The feature importance table shows that [write the top feature] was most useful for the Random Forest in this split. These are directional observations only, and they do not establish that the feature causes March sessions.

In [ ]:
forbidden_feature_words = [
    "march",
    "label",
    "future",
    "outcome"
]

print("Final model features:", list(X.columns))

assert list(X.columns) == feature_cols
assert "march_ga4_sessions" not in X.columns
assert not any(
    word in col.lower()
    for col in X.columns
    for word in forbidden_feature_words
)

print("Leakage check: PASS")

Final model features: ['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_sessions', 'ga4_engaged_sessions']
Leakage check: PASS


## Self-check

- The method choice is explained.
- The split is grouped by client and has no client overlap.
- The model and W04 baseline use the same test rows.
- The model and baseline use the same ranking metrics.
- February fields are used as features.
- March GA4 sessions is used only as the future label.
- Errors and feature importance are inspected.
- The assigned research paper was read and summarized.
- No client names, URLs, private queries, or future features were included.
- The notebook runs top to bottom without errors.
- Metrics were saved to `work/outputs/w05_model_metrics.json`.
- The notebook is committed under `work/notebooks/w05_model.ipynb`.